# Quantile Forecasting & Safety Stock — Smooth Demand

Uses the **best parameters found by Optuna** (loaded from `metadata.json`),
swapping only the `objective` from `regression` to `quantile`.

| Quantil | Uso |
|---|---|
| q50 | Point forecast (median) |
| q80 | Moderate stock |
| q95 | Conservative stock / safety stock |

**Safety Stock** = q95 - q50 (buffer above the median to guarantee the service level)

In [ ]:
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor

import sys
sys.path.insert(0, '../../src')
from config import LoadConfig
from pipeline.features import FeatureLoader, HOLIDAY_COLS, WEATHER_COLS, STATIC_COLS

warnings.filterwarnings('ignore')
os.makedirs('../../plots', exist_ok=True)

cfg    = LoadConfig()
loader = FeatureLoader(cfg)

## 1. Load Best Parameters from Training Artifact

In [ ]:
ARTIFACT_DIR = '../../artifacts/models/smooth'
with open(f'{ARTIFACT_DIR}/metadata.json') as f:
    meta = json.load(f)

BEST_PARAMS = meta['best_params']
FREQ        = meta['freq']
HORIZON     = meta['horizon']
TARGET_COL  = 'y'
SEED        = 42

print(f'Freq    : {FREQ}')
print(f'Horizon : {HORIZON} weeks')
print(f'Params  : {BEST_PARAMS}')

## 2. Load Data -- Same Split as Model Building

In [ ]:
df = loader.build_dataset('smooth')

cutoff   = df['ds'].max() - pd.Timedelta(weeks=HORIZON)
df_train = df[df['ds'] <= cutoff].copy()
df_test  = df[df['ds'] >  cutoff].copy()

STATIC_FEATURES = [c for c in STATIC_COLS if c in df_train.columns]

print(f'Cutoff : {cutoff.date()}')
print(f'Train  : {len(df_train):,} rows | {df_train["unique_id"].nunique():,} series')
print(f'Test   : {len(df_test):,} rows')

## 3. Configure Quantile LightGBM -- q50 / q80 / q95

Distinct subclasses so MLForecast generates separate output columns.

In [ ]:
LAG_PRESETS = {
    'short':  [4, 13],
    'medium': [4, 8, 13, 26],
    'long':   [4, 8, 13, 26, 52],
}
LAG_TF_MAP = {
    'rolling_mean_4':  RollingMean(window_size=4),
    'rolling_mean_13': RollingMean(window_size=13),
    'none':            None,
}

lags = LAG_PRESETS[BEST_PARAMS['lag_preset']]
lag_tf = LAG_TF_MAP[BEST_PARAMS['lag_transform']]
lag_transforms = {lag: [lag_tf] for lag in lags} if lag_tf else None

# Base parameters (strip structure keys, keep LightGBM params only)
lgbm_base = {
    k: v for k, v in BEST_PARAMS.items()
    if k not in ('lag_preset', 'lag_transform', 'target_transform')
}

# Subclasses to generate distinct columns in MLForecast
class LGBMq50(LGBMRegressor): pass
class LGBMq80(LGBMRegressor): pass
class LGBMq95(LGBMRegressor): pass

QUANTILES = [
    (LGBMq50, 0.50),
    (LGBMq80, 0.80),
    (LGBMq95, 0.95),
]

models = [
    cls(objective='quantile', alpha=alpha, random_state=SEED, verbose=-1, **lgbm_base)
    for cls, alpha in QUANTILES
]

mlf = MLForecast(
    models=models,
    freq=FREQ,
    lags=lags,
    lag_transforms=lag_transforms,
    target_transforms=[LocalStandardScaler()],
    date_features=['month', 'quarter', 'week'],
)

print(f'Lags    : {lags}')
print(f'Lag tf  : {BEST_PARAMS["lag_transform"]}')
print(f'Modelos : {[type(m).__name__ for m in models]}')

## 4. Train and Predict on Test Set

In [ ]:
mlf.fit(df_train, static_features=STATIC_FEATURES)
print('Fit complete')

In [ ]:
df_static   = loader.df_static
df_holidays = loader.df_holidays
df_weather  = loader.df_weather

avail_hol = [c for c in HOLIDAY_COLS if c in df_train.columns]
avail_wth = [c for c in WEATHER_COLS if c in df_train.columns]

future = (
    mlf.make_future_dataframe(h=HORIZON)
    .merge(df_static[['unique_id', 'region_id']], on='unique_id', how='left')
)
if avail_hol and df_holidays is not None:
    future = future.merge(
        df_holidays[['ds', 'region_id'] + avail_hol], on=['ds', 'region_id'], how='left'
    )
if avail_wth and df_weather is not None:
    future = future.merge(
        df_weather[['ds', 'region_id'] + avail_wth], on=['ds', 'region_id'], how='left'
    )

preds = mlf.predict(h=HORIZON, X_df=future.drop(columns='region_id'))

results = (
    df_test[['unique_id', 'ds', TARGET_COL]]
    .merge(preds, on=['unique_id', 'ds'], how='left')
)
results['safety_stock'] = results['LGBMq95'] - results['LGBMq50']

results.head()

## 5. Coverage Metrics and Pinball Loss

**Coverage**: % of weeks where the actual value fell below the predicted quantile (target = α itself).  
**Pinball Loss**: asymmetric metric -- penalises more when the actual exceeds the quantile.

In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    err = y_true - y_pred
    return float(np.mean(np.where(err >= 0, alpha * err, (alpha - 1) * err)))

q_cols  = ['LGBMq50', 'LGBMq80', 'LGBMq95']
alphas  = [0.50, 0.80, 0.95]

print(f'{"Model":<12} {"α":>5}  {"Pinball":>10}  {"Coverage":>10}  {"Target":>8}')
print('-' * 55)
for col, alpha in zip(q_cols, alphas):
    pl  = pinball_loss(results[TARGET_COL].values, results[col].values, alpha)
    cov = (results[TARGET_COL] <= results[col]).mean()
    print(f'{col:<12} {alpha:>5.2f}  {pl:>10.4f}  {cov:>9.1%}   {alpha:>7.0%}')

print(f'\nSafety Stock mean  : {results["safety_stock"].mean():.1f} units/series/week')
print(f'Safety Stock mediano: {results["safety_stock"].median():.1f}')

## 6. Visualisation -- Series by q95 Coverage

In [ ]:
N_HIST = 52

# Corporate blue palette
C_NAVY  = '#003D60'   # high coverage  — strong, authoritative
C_MID   = '#0A6EA3'   # median         — neutral
C_PALE  = '#7FA8C2'   # low coverage   — muted, signals weakness

# Only keep series that have training history (avoids empty hist in the plot)
train_ids = set(df_train['unique_id'].unique())

coverage_by_series = (
    results[results['unique_id'].isin(train_ids)]
    .groupby('unique_id')
    .apply(lambda g: (g[TARGET_COL] <= g['LGBMq95']).mean())
    .reset_index(name='coverage_q95')
    .sort_values('coverage_q95')
    .reset_index(drop=True)
)
n = len(coverage_by_series)

selected = pd.concat([
    coverage_by_series.tail(2).assign(quality='High coverage', color=C_NAVY),
    coverage_by_series.iloc[[n // 2]].assign(quality='Median',  color=C_MID),
    coverage_by_series.head(2).assign(quality='Low coverage',  color=C_PALE),
], ignore_index=True)

print(selected[['unique_id', 'coverage_q95', 'quality']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 5))

col_titles = [
    'High coverage #1', 'High coverage #2',
    'Median',
    'Low coverage #1', 'Low coverage #2',
]

for ax_idx, (_, row) in enumerate(selected.iterrows()):
    ax  = axes[ax_idx]
    uid = row['unique_id']
    cov = row['coverage_q95']
    c   = row['color']

    hist   = df_train[df_train['unique_id'] == uid].sort_values('ds').tail(N_HIST)
    actual = df_test[df_test['unique_id'] == uid].sort_values('ds')
    pred   = results[results['unique_id'] == uid].sort_values('ds')

    if hist.empty or actual.empty or pred.empty:
        ax.text(0.5, 0.5, f'No data
{uid}', transform=ax.transAxes,
                ha='center', va='center', fontsize=7)
        ax.set_title(col_titles[ax_idx], fontsize=8, fontweight='bold')
        continue

    cutoff = hist['ds'].max()

    # ── Quantile bands ────────────────────────────────────────────────────────
    ax.fill_between(pred['ds'], pred['LGBMq80'], pred['LGBMq95'],
                    alpha=0.15, color=c, label='q80-q95 band')
    ax.fill_between(pred['ds'], pred['LGBMq50'], pred['LGBMq80'],
                    alpha=0.30, color=c, label='q50-q80 band')

    # ── Series ────────────────────────────────────────────────────────────────
    ax.plot(hist['ds'], hist[TARGET_COL],
            color='#9BBFD4', linewidth=1.4, label='History')
    ax.plot([cutoff, actual['ds'].iloc[0]],
            [hist[TARGET_COL].iloc[-1], actual[TARGET_COL].iloc[0]],
            color='#9BBFD4', linewidth=1.4, linestyle='--')
    ax.plot(actual['ds'], actual[TARGET_COL],
            color='#003D60', linewidth=2, marker='o', markersize=5, label='Actual')
    ax.plot(pred['ds'], pred['LGBMq50'],
            color=c, linewidth=2, marker='s', markersize=4,
            linestyle='--', label='q50 (point)')
    ax.plot(pred['ds'], pred['LGBMq95'],
            color=c, linewidth=1.2, marker='^', markersize=4,
            linestyle=':', label='q95 (safety stock)')

    ax.axvline(cutoff, color='#003D60', linestyle=':', linewidth=0.8, alpha=0.35)

    ss_mean = pred['safety_stock'].mean()
    ax.set_title(f'{col_titles[ax_idx]}
Coverage q95={cov:.0%}  |  SS={ss_mean:.0f}u',
                 fontsize=8, fontweight='bold', color='#003D60')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%y'))
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(axis='y', alpha=0.2, color='#9BBFD4')

    if ax_idx == 0:
        ax.legend(fontsize=7, loc='upper left')

fig.suptitle('Quantile Forecasting — Safety Stock (Smooth Demand)',
             fontsize=13, fontweight='bold', color='#003D60', y=1.02)
plt.tight_layout()
plt.savefig('../../plots/quantile_safe_stock.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Safety Stock Distribution by Series

In [ ]:
ss_by_series = results.groupby('unique_id')['safety_stock'].mean()
ss_weekly    = results.groupby('ds')['safety_stock'].agg(['mean', 'median'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histogram ─────────────────────────────────────────────────────────────────
axes[0].hist(ss_by_series, bins=40, color='#005586', edgecolor='white', alpha=0.85)
for p, color, label in [
    (0.50, '#4DB4E0', 'p50'),
    (0.75, '#0A6EA3', 'p75'),
    (0.95, '#003D60', 'p95'),
]:
    val = ss_by_series.quantile(p)
    axes[0].axvline(val, color=color, linestyle='--', linewidth=1.8,
                    label=f'{label} = {val:.0f}u')
axes[0].set_title('Mean Safety Stock Distribution by Series',
                  fontweight='bold', color='#003D60')
axes[0].set_xlabel('Units  (q95 - q50)')
axes[0].set_ylabel('Number of series')
axes[0].legend(fontsize=9)

# ── By forecast week ─────────────────────────────────────────────────────────
x = range(len(ss_weekly))
axes[1].bar(x, ss_weekly['mean'], color='#005586', alpha=0.75, label='Mean')
axes[1].plot(x, ss_weekly['median'], color='#4DB4E0', marker='o',
             linewidth=2, markersize=6, label='Median')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'H+{i+1}' for i in x])
axes[1].set_title('Safety Stock by Forecast Week',
                  fontweight='bold', color='#003D60')
axes[1].set_ylabel('Units')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.grid(axis='y', alpha=0.2, color='#9BBFD4')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../../plots/safe_stock_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

print('
Safety Stock per series (units):')
print(ss_by_series.describe().round(1).to_string())